# 🔍 SOMEF Extraction Pipeline

This notebook runs the SOMEF tool on GitHub repositories to extract software metadata and to produce Codemeta JSON outputs.
**Important:** this is an extraction pipeline (SOMEF-based), not a model evaluation notebook. The main purpose is to run SOMEF, filter README-sourced fields, and save filtered Codemeta files for later evaluation or merging with other sources.

## Pipeline Overview:
1. **SOMEF Run** - Execute `somef describe` for each repository to generate SOMEF JSON and Codemeta outputs.
2. **Filter README-sourced Entities** - Keep only values that SOMEF attributes to the README (helps focus on README-evidenced metadata).
3. **Produce Filtered Codemeta** - Create and save Codemeta JSON files containing only README-sourced fields.

## Requirements
- **Python 3.9 or 3.10** — SOMEF requires Python 3.9/3.10; ensure your environment uses one of these versions.

## 1. Setup and Dependencies

In [1]:
# Install dependencies
!pip install tqdm somef --quiet

import re
import json
import os
import logging
import tempfile
import subprocess
from typing import Dict, Any,List

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger("SOMEF_EXTRACTION")


## 2. Configuration

In [2]:
# Evaluation configuration
OUTPUT_DIR = "../evaluation/extrated_codemeta_files"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 3. SOMEF Metadata Extraction

In [8]:
def run_somef_on_repository(repo_url: str) -> tuple[Dict[str, Any], Dict[str, Any]]:
    """Run SOMEF on GitHub repository and return both SOMEF and Codemeta results."""
    somef_data = {}
    codemeta_data = {}
    
    try:
        # Create temporary directory for SOMEF output
        with tempfile.TemporaryDirectory() as temp_dir:
            output_file = os.path.join(temp_dir, "somef_output.json")
            codemeta_file = os.path.join(temp_dir, "codemeta.json")
            
            # Run SOMEF command - separate output files for each format
            cmd = [
                "somef", "describe",
                "-r", repo_url,
                "-o", output_file,        # SOMEF JSON output
                "-c", codemeta_file,      # Codemeta output
                "-t", "0.8"  # Confidence threshold
            ]
            
            logger.info(f"Running SOMEF on {repo_url}...")
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
            
            if result.returncode != 0:
                logger.error(f"SOMEF failed: {result.stderr}")
                return {}, {}
            
            # Retrieve Codemeta output
            if os.path.exists(codemeta_file):
                with open(codemeta_file, 'r', encoding='utf-8') as f:
                    codemeta_data = json.load(f)
                logger.info("Codemeta extraction completed successfully")
                # Save Codemeta output for evaluation
                repo_name = repo_url.rstrip('/').split('/')[-1]
                codemeta_output_path = os.path.join(OUTPUT_DIR, f"000{repo_name}_codemeta.json")
                with open(codemeta_output_path, 'w', encoding='utf-8') as out_f:
                    json.dump(codemeta_data, out_f, indent=2)
                logger.info(f"Codemeta output saved to {codemeta_output_path}")
            else:
                logger.error("Codemeta output file not found")
            
            # Read SOMEF output
            if os.path.exists(output_file):
                with open(output_file, 'r', encoding='utf-8') as f:
                    somef_data = json.load(f)
                logger.info("SOMEF extraction completed successfully")
            else:
                logger.error("SOMEF output file not found")
            
            return somef_data, codemeta_data
                
    except subprocess.TimeoutExpired:
        logger.error("SOMEF execution timed out")
        return {}, {}
    except Exception as e:
        logger.error(f"Error running SOMEF: {e}")
        return {}, {}
    

def filter_somef_readme_metadata(somef_data: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    Extract all SOMEF entities where the source is the README.
    
    Returns a list of {field_name, value, confidence, source_url} for README-sourced entries.
    """
    readme_entities = []
    
    if not somef_data:
        return readme_entities
    
    # Iterate through all fields in SOMEF output
    for field_name, field_data in somef_data.items():
        # SOMEF returns lists for all field values
        if isinstance(field_data, list):
            for entry in field_data:
                if isinstance(entry, dict):
                    # Get source URL - it's a string, not a list
                    source = entry.get('source', '')
                    
                    # Only include if source is README
                    if isinstance(source, str) and 'readme' in source.lower():
                        # Extract the actual value/text
                        result_obj = entry.get('result', {})
                        text_value = None
                        
                        # Handle different result structures
                        if isinstance(result_obj, dict):
                            # Could be {'value': '...', 'type': '...'} or other formats
                            text_value = result_obj.get('value', '')
                        elif isinstance(result_obj, str):
                            text_value = result_obj
                        
                        # Fallback to excerpt if no result
                        if not text_value:
                            text_value = entry.get('excerpt', '')
                        
                        # Get confidence
                        confidence = entry.get('confidence', 1.0)
                        
                        if text_value:  # Only add if we have actual content
                            readme_entities.append({
                                'field': field_name,
                                'value': text_value,
                                'confidence': confidence,
                                'source': source,
                                'technique': entry.get('technique', ''),  # e.g., 'header_analysis', 'file_exploration'
                                'original_header': entry.get('result', {}).get('original_header', '') if isinstance(entry.get('result'), dict) else ''
                            })
    
    logger.info(f"Filtered {len(readme_entities)} README-sourced entities from SOMEF")
    return readme_entities

## 4. Filter Codemeta by README-sourced fields

In [14]:
def filter_codemeta_by_readme(codemeta_data: Dict[str, Any], 
                                somef_entities: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Dynamically filter codemeta.json to only include fields whose values 
    match or contain values from README-sourced SOMEF entities.
    For lists, only includes individual items that match README sources.
    Adds source information to track which README sources matched.
    
    Args:
        codemeta_data: The full codemeta.json data
        somef_entities: List of README-sourced entities from SOMEF
    
    Returns:
        Filtered codemeta dictionary with only README-sourced fields,
        including a '_source' metadata field for each value
    """
    # Build a mapping of values to their source information
    readme_value_sources = {}
    for entity in somef_entities:
        value = entity.get('value', '')
        if isinstance(value, str) and value.strip():
            normalized_value = value.strip()
            readme_value_sources[normalized_value] = {
                'source': entity.get('source', ''),
                'confidence': entity.get('confidence', 1.0),
                'technique': entity.get('technique', ''),
                'field': entity.get('field', '')
            }
    
    logger.info(f"Extracted {len(readme_value_sources)} unique values from README-sourced SOMEF entities")
    
    # Start with required codemeta fields
    filtered_codemeta = {
        "@context": codemeta_data.get("@context"),
        "@type": codemeta_data.get("@type")
    }
    
    def normalize_value(value):
        """Normalize a value for comparison."""
        if isinstance(value, str):
            return value.strip()
        return value
    
    def find_matching_sources(codemeta_value, readme_value_sources) -> List[Dict[str, Any]]:
        """
        Find all README sources that match a single codemeta value (string or dict).
        Returns list of source information dicts.
        """
        matched_sources = []
        
        if isinstance(codemeta_value, str):
            normalized = normalize_value(codemeta_value)
            
            # Direct match
            if normalized in readme_value_sources:
                matched_sources.append(readme_value_sources[normalized])
            else:
                # Check substring matches
                for readme_val, source_info in readme_value_sources.items():
                    if readme_val in normalized or normalized in readme_val:
                        matched_sources.append(source_info)
        
        elif isinstance(codemeta_value, dict):
            # Collect matches from all values in dict
            for v in codemeta_value.values():
                if isinstance(v, str):
                    matched_sources.extend(find_matching_sources(v, readme_value_sources))
        
        return matched_sources
    
    def filter_and_annotate_value(field_value, readme_value_sources):
        """
        Filter a value and return (filtered_value, sources) tuple.
        For lists, only keeps items that match README sources.
        Returns (None, []) if nothing matches.
        """
        if isinstance(field_value, list):
            # Filter list items individually
            filtered_items = []
            all_sources = []
            
            for item in field_value:
                item_sources = find_matching_sources(item, readme_value_sources)
                if item_sources:
                    filtered_items.append(item)
                    all_sources.extend(item_sources)
            
            if filtered_items:
                return filtered_items, all_sources
            else:
                return None, []
        
        else:
            # Handle strings and dicts
            sources = find_matching_sources(field_value, readme_value_sources)
            if sources:
                return field_value, sources
            else:
                return None, []
    
    # Filter codemeta fields dynamically and add source info
    for field_name, field_value in codemeta_data.items():
        # Skip already added required fields
        if field_name in ["@context", "@type"]:
            continue
        
        # Filter value and get matching sources
        filtered_value, matched_sources = filter_and_annotate_value(field_value, readme_value_sources)
        
        if filtered_value is not None and matched_sources:
            # Add the filtered field value
            filtered_codemeta[field_name] = filtered_value
            
            # Log what was filtered
            if isinstance(field_value, list) and isinstance(filtered_value, list):
                original_count = len(field_value)
                filtered_count = len(filtered_value)
                if original_count != filtered_count:
                    logger.info(f"Including field '{field_name}' - filtered list from {original_count} to {filtered_count} items (only README-sourced)")
                else:
                    logger.info(f"Including field '{field_name}' - all {filtered_count} list items matched README sources")
            else:
                logger.info(f"Including field '{field_name}' - matched {len(matched_sources)} README source(s)")
    
    return filtered_codemeta

## 5. Run batch extraction

In [15]:
REPO_URLS = [
    "https://github.com/qc2nl/qc2",
    "https://github.com/resurfemg-org/ReSurfEMG",
    "https://github.com/eWaterCycle/Cesium-NcWMS",
    "https://github.com/MindTheGap-ERC/admtools",
    "https://github.com/sanctuuary/APE",
    "https://github.com/computationalgeography/lue",
    "https://github.com/opensim-org/opensim-core",
    "https://github.com/NNPDF/nnpdf",
    "https://github.com/SMEISEN/AutoPQ",
    "https://github.com/iBridges-for-iRODS/iBridges-GUI"
]

for REPO_URL in REPO_URLS:
    # Run SOMEF extraction
    somef_data, codemeta_data = run_somef_on_repository(REPO_URL)

    somef_entities = filter_somef_readme_metadata(somef_data)
    logger.info(f"Extracted {somef_data}")

    # Filter codemeta to only README-sourced fields
    filtered_codemeta = filter_codemeta_by_readme(codemeta_data, somef_entities)

    # Save extracted Codemeta data
    repo_name = REPO_URL.rstrip('/').split('/')[-1]
    output_path = os.path.join(OUTPUT_DIR, f"{repo_name}_somef_codemeta.json")
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(filtered_codemeta, f, indent=2)
    logger.info(f"Saved Codemeta data to {output_path}")

logger.info("SOMEF extraction and filtering completed.")

2025-11-26 14:44:19,619 - INFO - Running SOMEF on https://github.com/qc2nl/qc2...
2025-11-26 14:44:30,099 - INFO - Codemeta extraction completed successfully
2025-11-26 14:44:30,100 - INFO - Codemeta output saved to ../evaluation/extrated_codemeta_files/000qc2_codemeta.json
2025-11-26 14:44:30,101 - INFO - SOMEF extraction completed successfully
2025-11-26 14:44:30,102 - INFO - Filtered 10 README-sourced entities from SOMEF
2025-11-26 14:44:30,102 - INFO - Extracted {'somef_provenance': {'somef_version': '0.9.12', 'somef_schema_version': '1.0.0', 'date': '2025-11-26 14:44:23'}, 'code_repository': [{'result': {'value': 'https://github.com/qc2nl/qc2', 'type': 'Url'}, 'confidence': 1, 'technique': 'GitHub_API'}], 'owner': [{'result': {'value': 'qc2nl', 'type': 'Organization'}, 'confidence': 1, 'technique': 'GitHub_API'}], 'date_created': [{'result': {'value': '2023-03-02T08:38:44Z', 'type': 'Date'}, 'confidence': 1, 'technique': 'GitHub_API'}], 'date_updated': [{'result': {'value': '2025-